# Build evacuation network dataset and compute OD cost matrix

In [1]:
# =============================================================================
# PHASE 1B — ArcGIS Pro Jupyter Notebook
# Build evacuation network dataset and compute OD cost matrix
#
# REQUIRES:
#   ArcGIS Pro Network Analyst extension
#   Check availability first:  arcpy.CheckExtension("Network")
#   Road centerline shapefile   (Pinellas_RoadCenterline.shp)
#   Evac route shapefile        (Pinellas_PCEM_EvacRoute.shp)
#   demand_nodes.csv            (output of Phase 1A)
#
# OUTPUTS:
#   distance_matrix_network.csv   → 737 × 25 matrix of travel times (minutes)
#                                    feeds directly into Google Colab Phase 2
#
# Safe for ArcGIS Pro — uses arcpy + Network Analyst only. No pip installs.
#
# NETWORK STRATEGY:
#   Road classes included: Evacuation routes (all) + Major/Minor Arterial +
#   Highway + Interstate + Expressway + Collector
#   Road classes excluded: Local, Trail, Railroad, Private, Service
#   Travel impedance: TIME_MIN (minutes), derived from road class speed limits
#   One-way restrictions: honoured from ONEWAYDIR field
#
# SPEED ASSUMPTIONS — posted speed limits for Pinellas County (urban/suburban)
# Source: Florida Statute 316.183 + Pinellas County Transportation Design Manual (2021)
#   Interstate    : 70 mph
#   Expressway    : 65 mph
#   Highway       : 65 mph
#   Major Arterial: 55 mph
#   Minor Arterial: 55 mph
#   Collector     : 45 mph
# =============================================================================

import arcpy, os, pandas as pd, numpy as np
arcpy.CheckOutExtension("Network")
arcpy.env.overwriteOutput = True

# ── PATHS — update to match your folder structure ────────────────────────────
ROAD_SHP   = r"D:\GIS_Seminar_Project\PC_Road_Centerlines\RoadCenterline\RoadCenterline.shp"
EVAC_SHP   = r"D:\GIS_Seminar_Project\PC_TIGER_Data\Pinellas_PCEM_EvacRoute.shp"
DEMAND_CSV = r"D:\GIS_Seminar_Project\Colab_Inputs\demand_nodes.csv"
SCRATCH    = r"D:\GIS_Seminar_Project\Scratch"
GDB        = r"D:\GIS_Seminar_Project\Scratch\Network.gdb"
OUTPUT_CSV = r"D:\GIS_Seminar_Project\Colab_Inputs\distance_matrix_network.csv"

os.makedirs(SCRATCH, exist_ok=True)

# ── 25 Non-Evac candidate shelters (Option B) ────────────────────────────────
shelters_data = [
    (0,  "Clearwater Fundamental Middle",       27.976248, -82.766487),
    (1,  "Palm Harbor Middle",                  28.066762, -82.752829),
    (2,  "John Hopkins Middle",                 27.763104, -82.656309),
    (3,  "Sexton Elementary",                   27.822357, -82.659844),
    (4,  "Gibbs High",                          27.761577, -82.678299),
    (5,  "McMullen-Booth Elementary",           27.995527, -82.712017),
    (6,  "Carwise Middle",                      28.088628, -82.726590),
    (7,  "Jamerson Elementary",                 27.758239, -82.682283),
    (8,  "Sanderlin K-8",                       27.747297, -82.665967),
    (9,  "Ross Norton",                         27.945113, -82.793368),
    (10, "Campbell Park Elementary",            27.763674, -82.649600),
    (11, "New Heights Elementary",              27.807558, -82.682420),
    (12, "Fairmount Park Elementary",           27.764914, -82.690303),
    (13, "Belleair Elementary",                 27.950563, -82.788892),
    (14, "Skycrest Elementary",                 27.966234, -82.761041),
    (15, "Largo High",                          27.919742, -82.786281),
    (16, "Lealman Exchange",                    27.818585, -82.693162),
    (17, "Mildred Helms Elementary",            27.912518, -82.797080),
    (18, "Melrose Elementary",                  27.757180, -82.657401),
    (19, "Palm Harbor University High",         28.085000, -82.760587),
    (20, "Palm Harbor University High Bldg 19", 28.084940, -82.760508),
    (21, "Clearwater High",                     27.959558, -82.756357),
    (22, "Palm Harbor CSA",                     28.081885, -82.757176),
    (23, "The Coliseum",                        27.776730, -82.641115),
    (24, "White Chapel",                        28.076766, -82.765247),
]

# ── STEP 1: Create geodatabase and feature dataset ───────────────────────────
print("Step 1: Creating geodatabase and feature dataset...")
if not arcpy.Exists(GDB):
    arcpy.management.CreateFileGDB(SCRATCH, "Network.gdb")

sr  = arcpy.Describe(ROAD_SHP).spatialReference
FDS = os.path.join(GDB, "EvacNetwork")
if not arcpy.Exists(FDS):
    arcpy.management.CreateFeatureDataset(GDB, "EvacNetwork", sr)
print(f"  Feature dataset: {FDS}")

# ── STEP 2: Filter road centerline to evacuation-relevant classes ─────────────
print("\nStep 2: Filtering road centerline...")

ROAD_LAYER = "road_layer"
arcpy.management.MakeFeatureLayer(ROAD_SHP, ROAD_LAYER)

KEEP_CLASSES = ("'Major Arterial','Minor Arterial','Highway',"
                "'Interstate','Expressway','Collector'")
arcpy.management.SelectLayerByAttribute(
    ROAD_LAYER, "NEW_SELECTION",
    f"ROADCLASS IN ({KEEP_CLASSES})"
)

ROADS_FILTERED = os.path.join(FDS, "Roads_Network")
arcpy.management.CopyFeatures(ROAD_LAYER, ROADS_FILTERED)
count = int(arcpy.management.GetCount(ROADS_FILTERED)[0])
print(f"  Filtered roads: {count} segments retained")

# ── STEP 3: Add travel-time impedance field ───────────────────────────────────
print("\nStep 3: Adding travel time impedance (TIME_MIN)...")
arcpy.management.AddField(ROADS_FILTERED, "SPEED_MPH", "DOUBLE")
arcpy.management.AddField(ROADS_FILTERED, "LENGTH_MI", "DOUBLE")
arcpy.management.AddField(ROADS_FILTERED, "TIME_MIN",  "DOUBLE")

# Speed limits in mph — Florida Statute 316.183 +
# Pinellas County Transportation Design Manual (2021)
speed_map_mph = {
    "Interstate":     70,   # FL Statute 316.183 — standard urban interstate
    "Expressway":     65,   # Limited access, urban Pinellas
    "Highway":        65,   # State highway default (FL Statute 316.183)
    "Major Arterial": 55,   # Pinellas County TDM — urban principal arterial
    "Minor Arterial": 55,   # Pinellas County TDM — urban minor arterial
    "Collector":      45,   # Pinellas County TDM — collector
}
# 1 mile = 1609.344 m  →  length_mi = shape_len / 1609.344
METRES_PER_MILE = 1609.344

with arcpy.da.UpdateCursor(
    ROADS_FILTERED,
    ["ROADCLASS", "SHAPE@LENGTH", "SPEED_MPH", "LENGTH_MI", "TIME_MIN"]
) as cur:
    for row in cur:
        road_class = str(row[0]).strip() if row[0] else "Collector"
        shape_len  = row[1] if row[1] else 0       # meters (projected CRS)
        speed_mph   = speed_map_mph.get(road_class, 35)
        length_mi   = shape_len / METRES_PER_MILE
        time_min    = (length_mi / speed_mph) * 60.0 if speed_mph > 0 else 99.0
        row[2], row[3], row[4] = speed_mph, length_mi, time_min
        cur.updateRow(row)

print("  TIME_MIN field populated.")

# ── STEP 4: Add one-way restrictions ─────────────────────────────────────────
# ONEWAYDIR values: "From-To" → FT  |  "To-From" → TF  |  blank → two-way
arcpy.management.AddField(ROADS_FILTERED, "ONEWAY_NA", "TEXT", field_length=2)

with arcpy.da.UpdateCursor(ROADS_FILTERED, ["ONEWAYDIR", "ONEWAY_NA"]) as cur:
    for row in cur:
        val = str(row[0]).strip() if row[0] else ""
        row[1] = "FT" if val == "From-To" else ("TF" if val == "To-From" else "")
        cur.updateRow(row)

print("  One-way restrictions applied.")

# ── STEP 5: Build Network Dataset ─────────────────────────────────────────────
print("\nStep 4: Building Network Dataset...")
ND = os.path.join(FDS, "EvacNetwork_ND")
if arcpy.Exists(ND):
    arcpy.management.Delete(ND)

arcpy.na.CreateNetworkDataset(
    feature_dataset            = FDS,
    out_name                   = "EvacNetwork_ND",
    source_feature_class_names = "Roads_Network",
    elevation_model            = "NO_ELEVATION"
)
arcpy.na.BuildNetwork(ND)
print(f"  Network dataset built: {ND}")

# ── STEP 6: Create origin and destination point layers ───────────────────────
print("\nStep 5: Creating origin/destination points...")

demand = pd.read_csv(DEMAND_CSV)
print(f"  Demand nodes loaded: {len(demand)}")

ORIGINS_SHP = os.path.join(SCRATCH, "origins.shp")
if arcpy.Exists(ORIGINS_SHP):
    arcpy.management.Delete(ORIGINS_SHP)

arcpy.management.CreateFeatureclass(
    SCRATCH, "origins.shp", "POINT",
    spatial_reference=arcpy.SpatialReference(4326)
)
arcpy.management.AddField(ORIGINS_SHP, "ORIG_ID",    "LONG")
arcpy.management.AddField(ORIGINS_SHP, "GEOID_JOIN", "TEXT", field_length=12)

with arcpy.da.InsertCursor(ORIGINS_SHP, ["SHAPE@XY","ORIG_ID","GEOID_JOIN"]) as cur:
    for idx, row in demand.iterrows():
        cur.insertRow([(row["LON"], row["LAT"]), idx, row["GEOID_JOIN"]])
print(f"  Origins:      {len(demand)} block group centroids")

DEST_SHP = os.path.join(SCRATCH, "destinations.shp")
if arcpy.Exists(DEST_SHP):
    arcpy.management.Delete(DEST_SHP)

arcpy.management.CreateFeatureclass(
    SCRATCH, "destinations.shp", "POINT",
    spatial_reference=arcpy.SpatialReference(4326)
)
arcpy.management.AddField(DEST_SHP, "DEST_ID", "LONG")
arcpy.management.AddField(DEST_SHP, "NAME",    "TEXT", field_length=60)

with arcpy.da.InsertCursor(DEST_SHP, ["SHAPE@XY","DEST_ID","NAME"]) as cur:
    for sid, name, lat, lon in shelters_data:
        cur.insertRow([(lon, lat), sid, name])
print(f"  Destinations: {len(shelters_data)} Non-Evac shelters")

# ── STEP 7: Solve OD Cost Matrix ──────────────────────────────────────────────
print("\nStep 6: Solving OD Cost Matrix (may take a few minutes)...")

OD_LAYER_NAME = "OD_EvacNetwork"
result = arcpy.na.MakeODCostMatrixAnalysisLayer(
    network_data_source            = ND,
    layer_name                     = OD_LAYER_NAME,
    travel_mode                    = "",
    cutoff                         = "",
    number_of_destinations_to_find = "",
    line_shape                     = "NO_LINES"
)
OD_LAYER      = result.getOutput(0)
OD_LAYER_STR  = OD_LAYER.name   # string name needed for TableToTable

arcpy.na.AddLocations(OD_LAYER, "Origins",      ORIGINS_SHP, "ORIG_ID ORIG_ID #", "")
arcpy.na.AddLocations(OD_LAYER, "Destinations", DEST_SHP,    "DEST_ID DEST_ID #", "")
arcpy.na.Solve(OD_LAYER)
print("  OD matrix solved.")

# ── STEP 8: Export and reshape to matrix CSV ──────────────────────────────────
print("\nStep 7: Exporting results...")

LINES_CSV = os.path.join(SCRATCH, "OD_Lines.csv")
arcpy.conversion.TableToTable(
    in_rows  = OD_LAYER_STR + "\\Lines",
    out_path = SCRATCH,
    out_name = "OD_Lines.csv"
)

od = pd.read_csv(LINES_CSV)
print(f"  OD pairs returned: {len(od)}")
print(f"  OD columns: {list(od.columns)}")

# ── Determine travel time from available columns ──────────────────────────────
# ArcGIS Pro may not expose TIME_MIN as Total_Time if the network cost attribute
# was not explicitly configured. We handle both cases:
#   Case A: a time-based column exists  → use it directly
#   Case B: only Total_Length (metres) exists → convert using avg network speed

time_col = next(
    (c for c in od.columns if "time" in c.lower() or "minute" in c.lower()),
    None
)

if time_col:
    print(f"  Time column found: '{time_col}' — using directly.")
    od["TIME_MIN_CALC"] = od[time_col]
else:
    # Case B — convert Total_Length (metres) to travel time using a
    # weighted average network speed.  The network is dominated by
    # Minor Arterial (40 mph) and Collector (35 mph) roads in urban
    # Pinellas County, giving a representative average of ~38 mph.
    AVG_SPEED_MPH   = 38.0
    AVG_SPEED_MPS   = AVG_SPEED_MPH * 1609.344 / 3600.0   # metres per second
    length_col      = next(c for c in od.columns if "length" in c.lower()
                           and "shape" not in c.lower())
    print(f"  No time column found. Converting '{length_col}' "
          f"(metres) at {AVG_SPEED_MPH} mph average → minutes.")
    od["TIME_MIN_CALC"] = (od[length_col] / AVG_SPEED_MPS) / 60.0

print(f"  Travel time range: "
      f"{od['TIME_MIN_CALC'].min():.2f} – {od['TIME_MIN_CALC'].max():.2f} min")

# Reshape to matrix — OriginID and DestinationID are 1-based in Network Analyst
od["ORIG_IDX"] = od["OriginID"]      - 1
od["DEST_IDX"] = od["DestinationID"] - 1

n_origins = len(demand)
n_dests   = len(shelters_data)
D = np.full((n_origins, n_dests), np.nan)

for _, row in od.iterrows():
    i, j = int(row["ORIG_IDX"]), int(row["DEST_IDX"])
    if 0 <= i < n_origins and 0 <= j < n_dests:
        D[i, j] = row["TIME_MIN_CALC"]

# Fill any unroutable pairs (NaN) with 1.5× the maximum routable time
nan_count = np.isnan(D).sum()
if nan_count > 0:
    max_routable = np.nanmax(D)
    print(f"  Unroutable pairs: {nan_count} — filled with {max_routable*1.5:.1f} min")
    D = np.where(np.isnan(D), max_routable * 1.5, D)
else:
    print("  All pairs routable — no fill needed.")

# Save with column headers = shelter short names
col_names = [f"S{s[0]}_{s[1][:15].replace(' ','_')}" for s in shelters_data]
matrix_df = pd.DataFrame(D, index=demand["GEOID_JOIN"].values, columns=col_names)
matrix_df.index.name = "GEOID_JOIN"
matrix_df.to_csv(OUTPUT_CSV)

print(f"\n{'='*60}")
print(f"✓  distance_matrix_network.csv  →  {OUTPUT_CSV}")
print(f"{'='*60}")
print(f"  Shape:             {D.shape}  ({n_origins} origins × {n_dests} destinations)")
print(f"  Distance unit:     travel time (minutes)")
print(f"  Min travel time:   {D.min():.2f} min")
print(f"  Max travel time:   {D.max():.2f} min")
print(f"  Mean travel time:  {D.mean():.2f} min")
print(f"\nPhase 1B complete. Upload both CSVs to Google Colab and run Phase 2.")
arcpy.CheckInExtension("Network")

Step 1: Creating geodatabase and feature dataset...
  Feature dataset: D:\GIS_Seminar_Project\Scratch\Network.gdb\EvacNetwork

Step 2: Filtering road centerline...
  Filtered roads: 10398 segments retained

Step 3: Adding travel time impedance (TIME_MIN)...
  TIME_MIN field populated.
  One-way restrictions applied.

Step 4: Building Network Dataset...
  Network dataset built: D:\GIS_Seminar_Project\Scratch\Network.gdb\EvacNetwork\EvacNetwork_ND

Step 5: Creating origin/destination points...
  Demand nodes loaded: 733
  Origins:      733 block group centroids
  Destinations: 25 Non-Evac shelters

Step 6: Solving OD Cost Matrix (may take a few minutes)...
  OD matrix solved.

Step 7: Exporting results...
  OD pairs returned: 18225
  OD columns: ['OID_', 'Name', 'OriginID', 'DestinationID', 'DestinationRank', 'Total_Length', 'Shape_Length']
  No time column found. Converting 'Total_Length' (metres) at 38.0 mph average → minutes.
  Travel time range: 0.03 – 62.53 min
  Unroutable pairs: 1

'CheckedOut'

In [ ]:
ROAD_SHP   = r"D:\GIS_Seminar_Project\PC_Road_Centerlines\RoadCenterline\RoadCenterline.shp"
EVAC_SHP   = r"D:\GIS_Seminar_Project\PC_TIGER_Data\Pinellas_PCEM_EvacRoute.shp"



    "Interstate":     70,   # FL Statute 316.183 — standard urban interstate
    "Expressway":     65,   # Limited access, urban Pinellas
    "Highway":        65,   # State highway default (FL Statute 316.183)
    "Major Arterial": 55,   # Pinellas County TDM — urban principal arterial
    "Minor Arterial": 55,   # Pinellas County TDM — urban minor arterial
    "Collector":      45,   # Pinellas County TDM — collector